# Plausible-value-aware explainable machine learning of creative thinking beyond academic scores
PISA 2022 Philippines · creative surplus · school-grouped, weighted, plausible-value-aware pipeline

## 1. Setup

In [ ]:
!pip install -q pyreadstat scikit-learn==1.6.1 xgboost==3.4.1 lightgbm==4.6.0 catboost==1.2.10 interpret==0.7.8 shap==0.54.1

In [ ]:
import os, glob, zipfile, json, time, pickle, platform, itertools, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kendalltau, loguniform, uniform, randint, t as t_dist
from sklearn.model_selection import ParameterSampler, KFold
from sklearn.metrics import (roc_auc_score, f1_score, balanced_accuracy_score, brier_score_loss,
                             precision_score, recall_score)
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

FOLDER = '/content/drive/MyDrive/PISA2022'
OUT_DIR = f'{FOLDER}/results'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs('/content/pisa2022', exist_ok=True)
for z in glob.glob(f'{FOLDER}/*.zip'):
    zipfile.ZipFile(z).extractall('/content/pisa2022')

def find(tag):
    hits = (glob.glob(f'/content/pisa2022/**/*{tag}*.[sS][aA][vV]', recursive=True)
            + glob.glob(f'{FOLDER}/*{tag}*.[sS][aA][vV]'))
    assert hits, f'{tag} not found'
    return hits[0]

STU_FILE, SCH_FILE, CRT_FILE = find('STU_QQQ'), find('SCH_QQQ'), find('CRT_COG')

## 2. Settings

In [ ]:
COUNTRY = 'PHL'
M, K, K_INNER = 10, 5, 3
N_CANDIDATES, N_BOOT, THRESH = 10, 200, 0.5
FOLD_SEED, TUNE_SEED, N_JOBS = 2022, 42, -1
INTER_MAX_ROWS = 400

PREDICTORS = {
    'Demographics': ['AGE', 'GRADE', 'ST004D01T', 'IMMIG', 'REPEAT'],
    'Family background': ['ESCS', 'HOMEPOS', 'PAREDINT', 'HISEI', 'ICTRES'],
    'Creativity-related': ['CREATAS', 'CREATOOS', 'CREATEFF', 'CREATOP', 'OPENART', 'CREATSCH', 'CREATFAM'],
    'Dispositions': ['CURIOAGR', 'GROSAGR'],
    'School climate and support': ['BELONG', 'BULLIED', 'FEELSAFE', 'SCHRISK', 'TEACHSUP', 'DISCLIM', 'FAMSUP'],
    'Learning and time use': ['MATHEFF', 'EXPOFA', 'EXPO21ST', 'SDLEFF', 'STUDYHMW', 'WORKPAY', 'WORKHOME',
                              'EXERPRAC', 'SKIPPING', 'TARDYSD'],
    'School characteristics': ['SCHLTYPE', 'SC001Q01TA', 'SCHSIZE', 'STRATIO', 'EDUSHORT', 'STAFFSHORT', 'CREACTIV'],
}
SCHOOL_VARS = PREDICTORS['School characteristics']
FEATURES = [v for vs in PREDICTORS.values() for v in vs]
SES_DEMO = ['ESCS', 'HOMEPOS', 'PAREDINT', 'HISEI', 'ICTRES', 'ST004D01T', 'IMMIG', 'SCHLTYPE']
CREATIVITY7 = PREDICTORS['Creativity-related']
TOP3 = ['SDLEFF', 'FEELSAFE', 'HOMEPOS']

MODELS = ['LightGBM', 'LR', 'XGBoost', 'CatBoost', 'RF', 'EBM']
EXPLAINED = ['LightGBM', 'RF', 'EBM', 'LR']
SEARCH_SPACES = {
    'LR': {'C': loguniform(1e-3, 1e2)},
    'RF': {'n_estimators': [300, 500, 800], 'max_depth': [4, 6, 8, 12, None],
           'min_samples_leaf': [5, 10, 20, 50], 'max_features': ['sqrt', 0.3, 0.5]},
    'XGBoost': {'n_estimators': randint(200, 801), 'learning_rate': loguniform(0.01, 0.1),
                'max_depth': randint(2, 7), 'subsample': uniform(0.6, 0.4),
                'colsample_bytree': uniform(0.5, 0.5), 'min_child_weight': [1, 5, 10],
                'reg_lambda': loguniform(0.1, 10)},
    'LightGBM': {'n_estimators': randint(200, 801), 'learning_rate': loguniform(0.01, 0.1),
                 'num_leaves': [7, 15, 31, 63], 'min_child_samples': [20, 50, 100],
                 'subsample': uniform(0.6, 0.4), 'colsample_bytree': uniform(0.5, 0.5),
                 'reg_lambda': loguniform(0.1, 10)},
    'CatBoost': {'iterations': randint(300, 1001), 'learning_rate': loguniform(0.01, 0.1),
                 'depth': randint(3, 8), 'l2_leaf_reg': uniform(1, 9)},
    'EBM': {'learning_rate': loguniform(0.005, 0.05), 'max_bins': [64, 128, 256],
            'interactions': [0, 5, 10], 'min_samples_leaf': [2, 10, 20]},
}

LABELS = {'AGE': 'Age', 'GRADE': 'Grade', 'ST004D01T': 'Female', 'IMMIG': 'Immigrant background',
          'REPEAT': 'Grade repetition', 'ESCS': 'Socioeconomic status (ESCS)', 'HOMEPOS': 'Home possessions',
          'PAREDINT': 'Parental education', 'HISEI': 'Parental occupation', 'ICTRES': 'ICT resources',
          'CREATAS': 'Creative activities at school', 'CREATOOS': 'Creative activities outside school',
          'CREATEFF': 'Creative self-efficacy', 'CREATOP': 'Creativity & openness to intellect',
          'OPENART': 'Openness to art & reflection', 'CREATSCH': 'Creative school & class environment',
          'CREATFAM': 'Creative peers & family environment', 'CURIOAGR': 'Curiosity', 'GROSAGR': 'Growth mindset',
          'BELONG': 'Sense of belonging', 'BULLIED': 'Exposure to bullying', 'FEELSAFE': 'Feeling safe at school',
          'SCHRISK': 'School safety risks', 'TEACHSUP': 'Teacher support (math)',
          'DISCLIM': 'Disciplinary climate (math)', 'FAMSUP': 'Family support', 'MATHEFF': 'Math self-efficacy',
          'EXPOFA': 'Exposure to formal & applied math', 'EXPO21ST': 'Exposure to math reasoning',
          'SDLEFF': 'Self-directed learning efficacy', 'STUDYHMW': 'Homework', 'WORKPAY': 'Paid work',
          'WORKHOME': 'Work at home', 'EXERPRAC': 'Exercise', 'SKIPPING': 'Skipping school',
          'TARDYSD': 'Arriving late', 'SCHLTYPE': 'Private school', 'SC001Q01TA': 'School location',
          'SCHSIZE': 'School size', 'STRATIO': 'Student-teacher ratio', 'EDUSHORT': 'Shortage of materials',
          'STAFFSHORT': 'Shortage of staff', 'CREACTIV': 'Creative extracurricular activities'}

BLUE, ORANGE = '#2a78d6', '#ee6a3a'
plt.rcParams.update({'font.family': 'serif', 'font.serif': ['Liberation Serif', 'Times New Roman', 'DejaVu Serif'],
                     'axes.spines.top': False, 'axes.spines.right': False, 'axes.edgecolor': '#444444',
                     'axes.grid': True, 'grid.color': '#e3e3e3', 'axes.axisbelow': True})
LOG = open(f'{OUT_DIR}/run_log.txt', 'a')

## 3. Functions

In [ ]:
def log(*a):
    s = ' '.join(str(x) for x in a); print(s); LOG.write(s + '\n'); LOG.flush()

def read_country(path, cols, chunksize=50000):
    import pyreadstat
    have = set(pyreadstat.read_sav(path, metadataonly=True)[1].column_names)
    missing = [c for c in cols if c not in have]
    assert not missing, f'variables not found in {path}: {missing}'
    use = list(dict.fromkeys(['CNT'] + cols))
    parts = [ch[ch['CNT'] == COUNTRY] for ch, _ in
             pyreadstat.read_file_in_chunks(pyreadstat.read_sav, path, chunksize=chunksize, usecols=use)]
    return pd.concat(parts, ignore_index=True)

def load_data():
    pv = lambda d: [f'PV{m}{d}' for m in range(1, M + 1)]
    stu = read_country(STU_FILE, ['CNTSCHID', 'CNTSTUID', 'W_FSTUWT'] + pv('MATH') + pv('READ') + pv('SCIE')
                       + [v for v in FEATURES if v not in SCHOOL_VARS])
    crt = read_country(CRT_FILE, ['CNTSCHID', 'CNTSTUID'] + pv('CRTH_NC'))
    sch = read_country(SCH_FILE, ['CNTSCHID'] + SCHOOL_VARS)
    df = (stu.merge(crt.drop(columns=['CNT', 'CNTSCHID']), on='CNTSTUID', how='inner')
             .merge(sch.drop(columns=['CNT']), on='CNTSCHID', how='left'))
    df = df[df[pv('CRTH_NC')].notna().all(axis=1) & (df['W_FSTUWT'] > 0)].reset_index(drop=True)
    df['ST004D01T'] = (df['ST004D01T'] == 1).astype(float)
    df['SCHLTYPE'] = df['SCHLTYPE'].map({1: 1.0, 2: 1.0, 3: 0.0})
    return df

def wquantile(x, w, p):
    o = np.argsort(x); x, w = x[o], w[o]
    cw = np.cumsum(w) / w.sum()
    return x[np.searchsorted(cw, p, side='left')]

def design(df, m, domains=('MATH', 'READ', 'SCIE'), poly=False):
    Z = np.column_stack([df[f'PV{m}{d}'].values for d in domains]) / 100.0
    cols = [Z]
    if poly:
        k = Z.shape[1]
        cols += [Z[:, [i]] * Z[:, [j]] for i in range(k) for j in range(i, k)]
    return np.column_stack([np.ones(len(df))] + cols)

def creative_surplus(df, m, domains=('MATH', 'READ', 'SCIE'), poly=False, fit_mask=None, thresholds=(1 / 3, 2 / 3)):
    X = design(df, m, domains, poly)
    c = df[f'PV{m}CRTH_NC'].values.astype(float)
    w = df['W_FSTUWT'].values.astype(float)
    f = np.ones(len(df), bool) if fit_mask is None else fit_mask
    Xw = X[f] * w[f, None]
    beta = np.linalg.solve(X[f].T @ Xw, Xw.T @ c[f])
    e = c - X @ beta
    mu = np.average(e[f], weights=w[f])
    sd = np.sqrt(np.average((e[f] - mu) ** 2, weights=w[f]))
    S = (e - mu) / sd
    r2 = 1 - np.average(e[f] ** 2, weights=w[f]) / np.average((c[f] - np.average(c[f], weights=w[f])) ** 2, weights=w[f])
    y = np.full(len(df), np.nan)
    if thresholds == 'median':
        q = wquantile(S[f], w[f], 0.5); y = (S > q).astype(float); qs = (q, q)
    else:
        qlo, qhi = wquantile(S[f], w[f], thresholds[0]), wquantile(S[f], w[f], thresholds[1])
        y[S >= qhi] = 1.0; y[S <= qlo] = 0.0; qs = (qlo, qhi)
    return S, y, qs, r2

def make_folds(df, k=K, seed=FOLD_SEED):
    schools = np.random.default_rng(seed).permutation(df['CNTSCHID'].unique())
    return df['CNTSCHID'].map({s: i % k for i, s in enumerate(schools)}).values

def build(name, params, seed=0):
    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier
    p = dict(params)
    if name == 'LR':
        return Pipeline([('imp', SimpleImputer(strategy='median', add_indicator=True)), ('sc', StandardScaler()),
                         ('clf', LogisticRegression(max_iter=2000, **p))])
    if name == 'RF':
        return Pipeline([('imp', SimpleImputer(strategy='median', add_indicator=True)),
                         ('clf', RandomForestClassifier(random_state=seed, n_jobs=N_JOBS, **p))])
    if name == 'XGBoost':
        from xgboost import XGBClassifier
        return XGBClassifier(tree_method='hist', eval_metric='logloss', random_state=seed, n_jobs=N_JOBS, **p)
    if name == 'LightGBM':
        from lightgbm import LGBMClassifier
        return LGBMClassifier(subsample_freq=1, random_state=seed, n_jobs=N_JOBS, verbose=-1, **p)
    if name == 'CatBoost':
        from catboost import CatBoostClassifier
        return CatBoostClassifier(verbose=0, random_seed=seed, thread_count=N_JOBS, **p)
    if name == 'EBM':
        from interpret.glassbox import ExplainableBoostingClassifier
        return ExplainableBoostingClassifier(outer_bags=8, random_state=seed, n_jobs=N_JOBS, **p)
    raise ValueError(name)

def fit(model, X, y, w):
    w = w / w.mean()
    if hasattr(model, 'steps'):
        model.fit(X, y, clf__sample_weight=w)
    else:
        model.fit(X, y, sample_weight=w)
    return model

def proba(model, X):
    return model.predict_proba(X)[:, 1]

def metrics(y, p, w):
    yh = (p >= THRESH).astype(int)
    return {'AUC': roc_auc_score(y, p, sample_weight=w), 'F1': f1_score(y, yh, sample_weight=w),
            'BalAcc': balanced_accuracy_score(y, yh, sample_weight=w), 'Brier': brier_score_loss(y, p, sample_weight=w),
            'Precision': precision_score(y, yh, sample_weight=w, zero_division=0),
            'Recall': recall_score(y, yh, sample_weight=w),
            'Specificity': recall_score(y, yh, pos_label=0, sample_weight=w)}

def cluster_boot(y, p, w, school, n_boot=N_BOOT, seed=0, only_auc=False):
    rng = np.random.default_rng(seed)
    codes = pd.factorize(school)[0]
    groups = np.split(np.argsort(codes, kind='stable'), np.cumsum(np.bincount(codes))[:-1])
    out = []
    for _ in range(n_boot):
        idx = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        if len(np.unique(y[idx])) < 2:
            continue
        out.append({'AUC': roc_auc_score(y[idx], p[idx], sample_weight=w[idx])} if only_auc
                   else metrics(y[idx], p[idx], w[idx]))
    return pd.DataFrame(out).var(ddof=1)

def rubin(est, var):
    est, var = np.asarray(est, float), np.asarray(var, float)
    Mm = len(est); qbar = est.mean(); U = var.mean(); B = est.var(ddof=1)
    T = U + (1 + 1 / Mm) * B
    df_ = (Mm - 1) * (1 + U / ((1 + 1 / Mm) * B)) ** 2 if B > 0 else np.inf
    h = t_dist.ppf(0.975, df_) * np.sqrt(T)
    return {'est': qbar, 'se': np.sqrt(T), 'lo': qbar - h, 'hi': qbar + h, 'df': df_,
            'between_share': (1 + 1 / Mm) * B / T if T > 0 else np.nan,
            'between_sd': np.sqrt(B), 'min': est.min(), 'max': est.max()}

def map_back(contrib, n_feat, imputer):
    out = np.abs(contrib[:, :n_feat]).copy()
    if imputer is not None and getattr(imputer, 'indicator_', None) is not None:
        for j, src in enumerate(imputer.indicator_.features_):
            out[:, src] += np.abs(contrib[:, n_feat + j])
    return out

def abs_contrib(name, model, X):
    p = X.shape[1]
    if name in ('LightGBM', 'XGBoost', 'CatBoost'):
        import shap
        sv = shap.TreeExplainer(model).shap_values(X)
        sv = sv[1] if isinstance(sv, list) else sv
        if sv.ndim == 3: sv = sv[:, :, 1]
        return np.abs(sv), sv
    if name == 'RF':
        import shap
        imp = model.named_steps['imp']; Xt = imp.transform(X)
        sv = shap.TreeExplainer(model.named_steps['clf']).shap_values(Xt)
        sv = sv[1] if isinstance(sv, list) else sv
        if sv.ndim == 3: sv = sv[:, :, 1]
        return map_back(sv, p, imp), None
    if name == 'LR':
        imp = model.named_steps['imp']; Z = model.named_steps['sc'].transform(imp.transform(X))
        return map_back(Z * model.named_steps['clf'].coef_[0], p, imp), None
    if name == 'EBM':
        T = model.eval_terms(X)
        out = np.zeros((X.shape[0], p))
        for t_idx, feats in enumerate(model.term_features_):
            for f in feats:
                out[:, f] += np.abs(T[:, t_idx]) / len(feats)
        return out, None
    raise ValueError(name)

def kendall_W(rank_matrix):
    R = np.asarray(rank_matrix, float); m_, n_ = R.shape
    S = ((R.sum(0) - R.sum(0).mean()) ** 2).sum()
    return 12 * S / (m_ ** 2 * (n_ ** 3 - n_))

def jaccard(a, b):
    a, b = set(a), set(b); return len(a & b) / len(a | b)

def top(s, k):
    return s.sort_values(ascending=False).index[:k]

def rank_of(imp):
    return imp.rank(ascending=False, method='average')

def ck_path(name):
    d = f'{OUT_DIR}/checkpoints'; os.makedirs(d, exist_ok=True); return f'{d}/{name}.pkl'

def ck_load(name):
    p = ck_path(name)
    if os.path.exists(p):
        with open(p, 'rb') as fh: return pickle.load(fh)
    return None

def cached(name, fn):
    obj = ck_load(name)
    if obj is not None: return obj
    t = time.perf_counter(); obj = fn(); obj['_seconds'] = time.perf_counter() - t
    with open(ck_path(name) + '.tmp', 'wb') as fh: pickle.dump(obj, fh)
    os.replace(ck_path(name) + '.tmp', ck_path(name))
    log(f"  saved {name} ({obj['_seconds'] / 60:.1f} min)")
    return obj

def to_json(o):
    return o.item() if hasattr(o, 'item') else str(o)

## 4. Data, creative-surplus target and folds

In [ ]:
T0 = time.perf_counter()
log(f"\n===== run started {time.strftime('%Y-%m-%d %H:%M:%S')} =====")
df = load_data()
X_all = df[FEATURES].astype(float).values
w_all = df['W_FSTUWT'].values.astype(float)
school = df['CNTSCHID'].values
folds = make_folds(df)
log(f'Analytic sample: {len(df)} students, {df.CNTSCHID.nunique()} schools')

Y, Q, R2 = {}, {}, {}
for m in range(1, M + 1):
    _, Y[m], Q[m], R2[m] = creative_surplus(df, m)
agree = []
for m in range(2, M + 1):
    both = ~np.isnan(Y[1]) & ~np.isnan(Y[m]); agree.append((Y[1][both] == Y[m][both]).mean())

rows = []
for k in range(K):
    fk = folds == k
    for m in range(1, M + 1):
        lab = fk & ~np.isnan(Y[m])
        rows.append({'fold': k + 1, 'PV': m, 'schools': len(np.unique(school[fk])), 'students': int(fk.sum()),
                     'labelled': int(lab.sum()), 'surplus_share_w': np.average(Y[m][lab], weights=w_all[lab])})
fold_tab = pd.DataFrame(rows); fold_tab.to_csv(f'{OUT_DIR}/folds_per_pv.csv', index=False)
thr_tab = pd.DataFrame({'PV': list(Q), 'q_1/3': [Q[m][0] for m in Q], 'q_2/3': [Q[m][1] for m in Q],
                        'R2': [R2[m] for m in Q], 'labelled': [int((~np.isnan(Y[m])).sum()) for m in Q]})
thr_tab.to_csv(f'{OUT_DIR}/thresholds_per_pv.csv', index=False)

results = {}
results['sample'] = {
    'students': len(df), 'schools': int(df.CNTSCHID.nunique()),
    'schools_per_fold': sorted(fold_tab.groupby('fold').schools.first().tolist()),
    'students_per_fold': sorted(fold_tab.groupby('fold').students.first().tolist()),
    'labelled_per_fold_range': [int(fold_tab.labelled.min()), int(fold_tab.labelled.max())],
    'surplus_share_range': [round(float(fold_tab.surplus_share_w.min()), 3), round(float(fold_tab.surplus_share_w.max()), 3)],
    'R2_mean': round(float(np.mean(list(R2.values()))), 3),
    'R2_range': [round(float(min(R2.values())), 3), round(float(max(R2.values())), 3)],
    'q_lo_mean': round(float(thr_tab['q_1/3'].mean()), 3), 'q_hi_mean': round(float(thr_tab['q_2/3'].mean()), 3),
    'labelled_mean': round(float(thr_tab.labelled.mean()), 1),
    'label_agreement_PV1_vs_others': round(float(np.mean(agree)), 3),
    'pct_observed': {v: round(float(df[v].notna().mean() * 100), 1) for v in FEATURES}}
log('Sample', {k: v for k, v in results['sample'].items() if k != 'pct_observed'})

## 5. Nested hyperparameter tuning (Table 2)

In [ ]:
best, tune_secs = {n: {} for n in MODELS}, {n: 0.0 for n in MODELS}
for n in MODELS:
    cands = list(ParameterSampler(SEARCH_SPACES[n], N_CANDIDATES, random_state=TUNE_SEED))
    for k in range(K):
        def run(n=n, k=k, cands=cands):
            tr = (folds != k) & ~np.isnan(Y[1])
            tr_sch = np.unique(school[tr]); rng = np.random.default_rng(TUNE_SEED + k)
            inner = dict(zip(rng.permutation(tr_sch), np.arange(len(tr_sch)) % K_INNER))
            ifold = np.array([inner[s] for s in school[tr]])
            Xtr, ytr, wtr = X_all[tr], Y[1][tr], w_all[tr]
            scores = []
            for c in cands:
                a = []
                for j in range(K_INNER):
                    it, iv = ifold != j, ifold == j
                    mdl = fit(build(n, c), Xtr[it], ytr[it], wtr[it])
                    a.append(roc_auc_score(ytr[iv], proba(mdl, Xtr[iv]), sample_weight=wtr[iv]))
                scores.append(np.mean(a))
            return {'params': cands[int(np.argmax(scores))], 'scores': scores}
        r = cached(f'tune_{n}_fold{k + 1}', run)
        best[n][k] = r['params']; tune_secs[n] += r['_seconds']
    log(f'tuned {n}: {best[n]}')
with open(f'{OUT_DIR}/TABLE2_selected_hyperparameters.json', 'w') as fh:
    json.dump({n: {f'fold {k + 1}': best[n][k] for k in range(K)} for n in MODELS}, fh, indent=1, default=to_json)

## 6. Training, prediction and explanation for every model and plausible value

In [ ]:
MAIN, OOF, IMP, per_pv = {}, {}, {}, []
for n in MODELS:
    MAIN[n], OOF[n], IMP[n] = {}, {}, {}
    for m in range(1, M + 1):
        def run(n=n, m=m):
            lab = ~np.isnan(Y[m]); p = np.full(len(df), np.nan)
            absc = np.full((len(df), len(FEATURES)), np.nan)
            signed = np.full((len(df), len(FEATURES)), np.nan, dtype=np.float32) if n == 'LightGBM' else None
            t_fit = t_shap = 0.0
            for k in range(K):
                tr, te = lab & (folds != k), lab & (folds == k)
                t = time.perf_counter()
                mdl = fit(build(n, best[n][k], seed=m * 10 + k), X_all[tr], Y[m][tr], w_all[tr])
                p[te] = proba(mdl, X_all[te]); t_fit += time.perf_counter() - t
                if n in EXPLAINED:
                    t = time.perf_counter()
                    a, s = abs_contrib(n, mdl, X_all[te]); absc[te] = a
                    if signed is not None and s is not None: signed[te] = s
                    t_shap += time.perf_counter() - t
            t = time.perf_counter()
            var = cluster_boot(Y[m][lab], p[lab], w_all[lab], school[lab], seed=m)
            t_boot = time.perf_counter() - t
            imp = pd.Series(np.nanmean(absc[lab], axis=0), index=FEATURES) if not np.isnan(absc).all() else None
            met = metrics(Y[m][lab], p[lab], w_all[lab])
            log(f"{n} PV{m}: AUC {met['AUC']:.3f}")
            return {'p': p, 'imp': imp, 'signed': signed, 'met': met, 'var': var.to_dict(),
                    't_fit': t_fit, 't_shap': t_shap, 't_boot': t_boot}
        r = cached(f'main_{n}_PV{m}', run)
        MAIN[n][m] = r; OOF[n][m] = r['p']
        if r['imp'] is not None: IMP[n][m] = r['imp']
        per_pv.append({'model': n, 'PV': m, **r['met'], **{f'var_{k_}': v for k_, v in r['var'].items()},
                       't_fit': r['t_fit'], 't_shap': r['t_shap'], 't_boot': r['t_boot']})
pv_df = pd.DataFrame(per_pv); pv_df.to_csv(f'{OUT_DIR}/per_pv_metrics.csv', index=False)

## 7. Pooled performance (Tables 4 and 5)

In [ ]:
rows = []
for n in MODELS:
    d = pv_df[pv_df.model == n]
    for k_ in ['AUC', 'F1', 'BalAcc', 'Brier', 'Precision', 'Recall', 'Specificity']:
        rows.append({'model': n, 'metric': k_, **rubin(d[k_], d[f'var_{k_}'])})
pooled_metrics = pd.DataFrame(rows); pooled_metrics.to_csv(f'{OUT_DIR}/pooled_metrics.csv', index=False)
get = lambda n, k_: pooled_metrics[(pooled_metrics.model == n) & (pooled_metrics.metric == k_)].iloc[0]

tab4 = pd.DataFrame({n: {**{k_: f"{get(n, k_)['est']:.3f} ± {get(n, k_)['se']:.3f}" for k_ in ['AUC', 'F1', 'BalAcc', 'Brier']},
                         'Between-PV SD of AUC': f"{get(n, 'AUC')['between_sd']:.3f}"} for n in MODELS}).T
tab4 = tab4.loc[sorted(MODELS, key=lambda n: -get(n, 'AUC')['est'])]
tab4.to_csv(f'{OUT_DIR}/TABLE4_performance.csv')
tab5 = pd.DataFrame({n: {k_: f"{get(n, k_)['est']:.3f} [{get(n, k_)['lo']:.3f}, {get(n, k_)['hi']:.3f}]"
                         for k_ in ['Precision', 'Recall', 'Specificity', 'AUC']} for n in tab4.index}).T
tab5.to_csv(f'{OUT_DIR}/TABLE5_threshold_metrics.csv')
results['table4'] = tab4.to_dict('index'); results['table5'] = tab5.to_dict('index')
results['between_share_AUC'] = {n: round(float(get(n, 'AUC')['between_share']), 3) for n in MODELS}
results['lightgbm_single_pv_auc_range'] = [round(float(get('LightGBM', 'AUC')['min']), 3),
                                          round(float(get('LightGBM', 'AUC')['max']), 3)]
log('\nTable 4\n' + tab4.to_string() + '\n\nTable 5\n' + tab5.to_string())

## 8. Ranking stability and cross-model agreement (Table 6)

In [ ]:
imp_l = pd.DataFrame(IMP['LightGBM'])
pooled_main = imp_l.mean(axis=1).sort_values(ascending=False)
pooled_main.to_csv(f'{OUT_DIR}/lightgbm_pooled_importance.csv')
pairs = list(itertools.combinations(imp_l.columns, 2))
results['stability'] = {
    'kendall_W': round(kendall_W(imp_l.rank(ascending=False).T.values), 3),
    'mean_tau': round(float(np.mean([kendalltau(imp_l[a], imp_l[b])[0] for a, b in pairs])), 3),
    'jaccard_top5': round(float(np.mean([jaccard(top(imp_l[a], 5), top(imp_l[b], 5)) for a, b in pairs])), 3),
    'jaccard_top10': round(float(np.mean([jaccard(top(imp_l[a], 10), top(imp_l[b], 10)) for a, b in pairs])), 3),
    'top5': list(pooled_main.index[:5])}
log('Stability', results['stability'])

pooled = {n: pd.DataFrame(IMP[n]).mean(axis=1) for n in EXPLAINED}
cross = pd.DataFrame([{'pair': f'{a} vs {b}', 'kendall_tau': kendalltau(pooled[a], pooled[b])[0],
                       'jaccard_top5': jaccard(top(pooled[a], 5), top(pooled[b], 5)),
                       'jaccard_top10': jaccard(top(pooled[a], 10), top(pooled[b], 10))}
                      for a, b in itertools.combinations(pooled, 2)])
cross.to_csv(f'{OUT_DIR}/TABLE6_cross_model_agreement.csv', index=False)
cross_ranks = pd.DataFrame({n: rank_of(pooled[n]) for n in pooled}).loc[list(pooled_main.index[:10])]
cross_ranks.to_csv(f'{OUT_DIR}/cross_model_ranks_of_lightgbm_top10.csv')
results['cross_model'] = {'pairs': cross.round(3).to_dict('records'), 'ranks': cross_ranks.round(1).to_dict('index')}
log('\nCross-model agreement\n' + cross.round(3).to_string() + '\n' + cross_ranks.to_string())

## 9. Figure 3: pooled SHAP importance

In [ ]:
ranks = imp_l.rank(ascending=False)
top12 = list(pooled_main.index[:12])[::-1]
fig, ax = plt.subplots(figsize=(8.2, 4.2))
ax.barh([LABELS[f] for f in top12], pooled_main[top12], color=BLUE, height=0.6)
ax.errorbar(pooled_main[top12], range(12), xerr=[pooled_main[top12] - imp_l.loc[top12].min(axis=1),
            imp_l.loc[top12].max(axis=1) - pooled_main[top12]], fmt='none', ecolor='black', capsize=3, lw=1.4)
ax.grid(axis='y', visible=False); ax.tick_params(axis='y', length=0, labelsize=12); ax.tick_params(axis='x', labelsize=11)
ax.set_xlabel('Mean |SHAP| (log-odds), pooled over PVs; whiskers = min–max across PVs', fontsize=12)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/FIG3_shap_importance.png', dpi=300); plt.show()
results['figure3'] = {LABELS[f]: {'pooled': round(float(pooled_main[f]), 3), 'min': round(float(imp_l.loc[f].min()), 3),
                                  'max': round(float(imp_l.loc[f].max()), 3),
                                  'rank_range': [int(ranks.loc[f].min()), int(ranks.loc[f].max())]}
                      for f in pooled_main.index}
results['always_top10'] = [f for f in pooled_main.index if ranks.loc[f].max() <= 10]

## 10. Figure 4: SHAP dependence

In [ ]:
shap_mean = np.nanmean(np.stack([MAIN['LightGBM'][m]['signed'] for m in range(1, M + 1)]).astype(float), axis=0)
fig, axes = plt.subplots(1, 3, figsize=(9.3, 2.9))
dep = {}
for ax, f in zip(axes, list(pooled_main.index[:3])):
    j = FEATURES.index(f); x, y = X_all[:, j], shap_mean[:, j]; ok = ~np.isnan(x) & ~np.isnan(y)
    ax.scatter(x[ok], y[ok], s=6, color=BLUE, alpha=0.35, lw=0)
    qs = np.unique(np.nanquantile(x[ok], np.linspace(0.05, 0.95, 11)))
    mids, means = [], []
    for a, b in zip(qs[:-1], qs[1:]):
        sel = ok & (x >= a) & (x <= b)
        if sel.sum() > 20: mids.append(np.median(x[sel])); means.append(y[sel].mean())
    ax.plot(mids, means, color=ORANGE, lw=2.6); ax.axhline(0, color='black', lw=1); ax.set_xlabel(LABELS[f], fontsize=12)
    dep[f] = {'binned_x': [round(float(v), 2) for v in mids], 'binned_mean_shap': [round(float(v), 3) for v in means]}
axes[0].set_ylabel('SHAP value (log-odds)', fontsize=12)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/FIG4_shap_dependence.png', dpi=300); plt.show()
results['figure4'] = dep

## 11. SHAP interaction values and EBM shape functions

In [ ]:
import shap
inter_rows = []
for m in range(1, M + 1):
    def run(m=m):
        lab = ~np.isnan(Y[m]); rows_ = []
        for k in range(K):
            tr, te = lab & (folds != k), lab & (folds == k)
            mdl = fit(build('LightGBM', best['LightGBM'][k], seed=m * 10 + k), X_all[tr], Y[m][tr], w_all[tr])
            te_idx = np.flatnonzero(te)
            if len(te_idx) > INTER_MAX_ROWS:
                te_idx = np.random.default_rng(m * 10 + k).choice(te_idx, INTER_MAX_ROWS, replace=False)
            iv = shap.TreeExplainer(mdl).shap_interaction_values(X_all[te_idx])
            iv = iv[1] if isinstance(iv, list) else iv
            for f in TOP3:
                i = FEATURES.index(f); row = np.abs(iv[:, i, :]).mean(0)
                rows_.append({'PV': m, 'fold': k, 'feature': f, 'interaction_share': (row.sum() - row[i]) / row.sum(),
                              **{FEATURES[j]: row[j] for j in range(len(FEATURES)) if j != i}})
        return {'rows': rows_}
    inter_rows += cached(f'interactions_PV{m}', run)['rows']
inter = pd.DataFrame(inter_rows); inter.to_csv(f'{OUT_DIR}/shap_interactions.csv', index=False)
inter_res = {}
for f in TOP3:
    d = inter[inter.feature == f].drop(columns=['PV', 'fold', 'feature']).mean()
    share = d.pop('interaction_share')
    inter_res[f] = {'interaction_share': round(float(share), 3),
                    'top_partners': d.sort_values(ascending=False).head(3).round(4).to_dict()}

def ebm_shapes():
    out = {}
    for m in (1, 4, 7, 10):
        lab = ~np.isnan(Y[m]); tr = lab & (folds != 0)
        mdl = fit(build('EBM', best['EBM'][0], seed=m), X_all[tr], Y[m][tr], w_all[tr])
        for f in TOP3:
            i = FEATURES.index(f)
            grid = np.nanquantile(df[f], np.linspace(0.02, 0.98, 40))
            Xg = np.tile(np.nanmedian(X_all, axis=0), (len(grid), 1)); Xg[:, i] = grid
            T = mdl.eval_terms(Xg)
            t_main = [t for t, fs in enumerate(mdl.term_features_) if tuple(fs) == (i,)][0]
            out[(m, f)] = (grid, T[:, t_main])
    return {'curves': out}
curves = cached('ebm_shapes', ebm_shapes)['curves']
fig, axes = plt.subplots(1, 3, figsize=(10, 2.8))
for ax, f in zip(axes, TOP3):
    cs = [curves[(m, f)] for m in (1, 4, 7, 10)]; grid = cs[0][0]
    for g, c in cs: ax.plot(g, c, lw=0.8, alpha=0.6)
    mean_curve = np.mean([c for _, c in cs], axis=0)
    ax.plot(grid, mean_curve, color='k', lw=1.6); ax.set_title(LABELS[f], fontsize=9)
    ax.set_xlabel('index value', fontsize=8); ax.set_ylabel('EBM score (log-odds)', fontsize=8)
    dd = np.sign(np.diff(np.convolve(mean_curve, np.ones(5) / 5, mode='valid'))); dd = dd[dd != 0]
    inter_res[f]['ebm_slope_sign_changes'] = int((np.diff(dd) != 0).sum())
    inter_res[f]['ebm_argmin_value'] = round(float(grid[np.argmin(mean_curve)]), 2)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/ebm_shape_functions.png', dpi=300); plt.show()
results['interactions_ebm'] = inter_res
log('Interactions and EBM shapes', json.dumps(inter_res, indent=1, default=to_json))

## 12. Sensitivity analysis (Table 6)

In [ ]:
base = lambda **kw: (lambda m, fit_mask=None: creative_surplus(df, m, fit_mask=fit_mask, **kw))
scenarios = [
    ('main', 'Main specification', base(), FEATURES, False),
    ('res_mr', 'Residual: math and reading only', base(domains=('MATH', 'READ')), FEATURES, False),
    ('res_quad', 'Residual: quadratic expectation', base(poly=True), FEATURES, False),
    ('res_fold', 'Residual: fold-internal estimation', base(), FEATURES, True),
    ('thr_q4', 'Threshold: top vs bottom quartile', base(thresholds=(0.25, 0.75)), FEATURES, False),
    ('thr_q5', 'Threshold: top vs bottom quintile', base(thresholds=(0.2, 0.8)), FEATURES, False),
    ('thr_med', 'Threshold: median split', base(thresholds='median'), FEATURES, False),
    ('p_noses', 'Predictors: without socioeconomic/demographic', base(), [f for f in FEATURES if f not in SES_DEMO], False),
    ('p_nocre', 'Predictors: without creativity indices', base(), [f for f in FEATURES if f not in CREATIVITY7], False),
    ('p_cre', 'Predictors: creativity indices + curiosity only', base(), CREATIVITY7 + ['CURIOAGR'], False),
    ('p_nosdl', 'Predictors: without self-directed learning efficacy', base(), [f for f in FEATURES if f != 'SDLEFF'], False),
]
rows = []
for key, name, fn, feats, fold_internal in scenarios:
    Xs = df[feats].astype(float).values; est, var, imps = [], [], []
    for m in range(1, M + 1):
        def run(fn=fn, feats=feats, fi=fold_internal, m=m, Xs=Xs):
            p = np.full(len(df), np.nan); yv = np.full(len(df), np.nan)
            absc = np.full((len(df), len(feats)), np.nan)
            y_m = None if fi else fn(m)[1]
            for k in range(K):
                y_k = fn(m, fit_mask=(folds != k))[1] if fi else y_m
                lab = ~np.isnan(y_k); tr, te = lab & (folds != k), lab & (folds == k)
                mdl = fit(build('LightGBM', best['LightGBM'][k], seed=m * 10 + k), Xs[tr], y_k[tr], w_all[tr])
                p[te] = proba(mdl, Xs[te]); yv[te] = y_k[te]
                absc[te] = abs_contrib('LightGBM', mdl, Xs[te])[0]
            lab = ~np.isnan(yv)
            return {'auc': roc_auc_score(yv[lab], p[lab], sample_weight=w_all[lab]),
                    'var': cluster_boot(yv[lab], p[lab], w_all[lab], school[lab], seed=m, only_auc=True)['AUC'],
                    'imp': pd.Series(np.nanmean(absc[lab], axis=0), index=feats)}
        r = cached(f'sens_{key}_PV{m}', run)
        est.append(r['auc']); var.append(r['var']); imps.append(r['imp'])
    rr = rubin(est, var); imp = pd.concat(imps, axis=1).mean(axis=1)
    common = [f for f in feats if f in pooled_main.index]
    rows.append({'specification': name, 'AUC': rr['est'], 'SE': rr['se'],
                 'kendall_tau_vs_main': kendalltau(imp[common], pooled_main[common])[0] if len(common) > 2 else np.nan,
                 'jaccard_top5_vs_main': jaccard(top(imp, 5), top(pooled_main, 5)), 'top5': ', '.join(top(imp, 5))})
    log(name, f"AUC {rr['est']:.3f} ± {rr['se']:.3f}")
sens = pd.DataFrame(rows); sens.to_csv(f'{OUT_DIR}/TABLE6_sensitivity.csv', index=False)
results['sensitivity'] = sens.round(3).to_dict('records')

## 13. Subgroup performance and fairness audit (Table 7)

In [ ]:
esc = df['ESCS'].values; ok = ~np.isnan(esc)
cuts = [wquantile(esc[ok], w_all[ok], q_) for q_ in (0.25, 0.5, 0.75)]
groups = {'Female': df['ST004D01T'].values == 1, 'Male': df['ST004D01T'].values == 0,
          'Public school': df['SCHLTYPE'].values == 0, 'Private school': df['SCHLTYPE'].values == 1,
          'ESCS Q1 (lowest)': ok & (esc <= cuts[0]), 'ESCS Q2': ok & (esc > cuts[0]) & (esc <= cuts[1]),
          'ESCS Q3': ok & (esc > cuts[1]) & (esc <= cuts[2]), 'ESCS Q4 (highest)': ok & (esc > cuts[2])}
rows = []
for m in range(1, M + 1):
    lab = ~np.isnan(Y[m]); p = OOF['LightGBM'][m]
    for g, gm in groups.items():
        s = lab & gm; y, w, ph = Y[m][s], w_all[s], (p[s] >= THRESH).astype(int)
        rows.append({'PV': m, 'group': g, 'weight_share': w.sum() / w_all[lab].sum(),
                     'observed_rate': np.average(y, weights=w), 'predicted_rate': np.average(ph, weights=w),
                     'AUC': roc_auc_score(y, p[s], sample_weight=w) if len(np.unique(y)) > 1 else np.nan,
                     'TPR': np.average(ph[y == 1], weights=w[y == 1]), 'FPR': np.average(ph[y == 0], weights=w[y == 0])})
fair_pv = pd.DataFrame(rows); fair_pv.to_csv(f'{OUT_DIR}/fairness_per_pv.csv', index=False)
tab7 = fair_pv.groupby('group', sort=False).mean(numeric_only=True).drop(columns='PV')
tab7.round(3).to_csv(f'{OUT_DIR}/TABLE7_fairness.csv')
gaps = {}
for attr, gs in {'gender': ['Female', 'Male'], 'school type': ['Public school', 'Private school'],
                 'ESCS': [g for g in groups if g.startswith('ESCS')]}.items():
    t_ = tab7.loc[gs]
    gaps[attr] = {'TPR_gap': round(float(t_.TPR.max() - t_.TPR.min()), 3),
                  'FPR_gap': round(float(t_.FPR.max() - t_.FPR.min()), 3),
                  'AUC_gap': round(float(t_.AUC.max() - t_.AUC.min()), 3)}
results['table7'] = tab7.round(3).to_dict('index'); results['equalized_odds_gaps'] = gaps
log('\nTable 7\n' + tab7.round(3).to_string() + '\n' + str(gaps))

## 14. Ablation of pipeline shortcuts (Table 8)

In [ ]:
random_folds = np.empty(len(df), int)
for k, (_, te) in enumerate(KFold(K, shuffle=True, random_state=FOLD_SEED).split(np.zeros(len(df)))):
    random_folds[te] = k
df_avg = df.copy()
for dom in ['MATH', 'READ', 'SCIE', 'CRTH_NC']:
    df_avg[f'PV1{dom}'] = df_avg[[f'PV{m}{dom}' for m in range(1, M + 1)]].mean(axis=1)
Y_avg = creative_surplus(df_avg, 1)[1]

def run_cfg(y, fold_vec, weighted, seed):
    lab = ~np.isnan(y); p = np.full(len(df), np.nan)
    for k in range(K):
        tr, te = lab & (fold_vec != k), lab & (fold_vec == k)
        wt = w_all[tr] if weighted else np.ones(tr.sum())
        p[te] = proba(fit(build('LightGBM', best['LightGBM'][k], seed=seed * 10 + k), X_all[tr], y[tr], wt), X_all[te])
    return {'auc': roc_auc_score(y[lab], p[lab], sample_weight=w_all[lab]),
            'var': cluster_boot(y[lab], p[lab], w_all[lab], school[lab], seed=seed, only_auc=True)['AUC']}

rows = []
for pv_mode in ['PV1 only', 'Averaged PVs', 'All ten pooled']:
    for split in ['Random', 'School-grouped']:
        for weighted in [False, True]:
            fv = folds if split == 'School-grouped' else random_folds
            key = f"abl_{pv_mode.split()[0]}_{split[:6]}_{'w' if weighted else 'nw'}"
            if pv_mode == 'All ten pooled' and split == 'School-grouped' and weighted:
                rr = rubin([MAIN['LightGBM'][m]['met']['AUC'] for m in range(1, M + 1)],
                           [MAIN['LightGBM'][m]['var']['AUC'] for m in range(1, M + 1)])
                auc, se = rr['est'], rr['se']
            elif pv_mode == 'All ten pooled':
                res = [cached(f'{key}_PV{m}', lambda m=m, fv=fv, weighted=weighted: run_cfg(Y[m], fv, weighted, m))
                       for m in range(1, M + 1)]
                rr = rubin([r['auc'] for r in res], [r['var'] for r in res]); auc, se = rr['est'], rr['se']
            else:
                y = Y[1] if pv_mode == 'PV1 only' else Y_avg
                r = cached(key, lambda y=y, fv=fv, weighted=weighted: run_cfg(y, fv, weighted, 1))
                auc, se = r['auc'], np.sqrt(r['var'])
            rows.append({'PV handling': pv_mode, 'Validation split': split,
                         'Training weights': 'Final weight' if weighted else 'None', 'AUC': auc, 'SE': se})
            log(pv_mode, split, rows[-1]['Training weights'], f'{auc:.3f} ± {se:.3f}')
tab8 = pd.DataFrame(rows)
proposed = tab8.iloc[-1]['AUC']
tab8['AUC ± SE'] = [f'{a:.3f} ± {s:.3f}' for a, s in zip(tab8.AUC, tab8.SE)]
tab8['ΔAUC vs proposed'] = [f'{a - proposed:+.3f}' for a in tab8.AUC]
tab8 = tab8[['PV handling', 'Validation split', 'Training weights', 'AUC ± SE', 'ΔAUC vs proposed']]
tab8.to_csv(f'{OUT_DIR}/TABLE8_ablation.csv', index=False)
results['table8'] = tab8.to_dict('records')
log('\nTable 8\n' + tab8.to_string())

## 15. Computational cost and software versions

In [ ]:
model_secs = {n: float(pv_df[pv_df.model == n][['t_fit', 't_shap']].sum().sum()) for n in MODELS}
all_ck = [ck_load(os.path.basename(p_)[:-4]) for p_ in glob.glob(f'{OUT_DIR}/checkpoints/*.pkl')]
total = sum(c.get('_seconds', 0) for c in all_ck if isinstance(c, dict))
try:
    import psutil; ram = f'{psutil.virtual_memory().total / 2**30:.0f} GB RAM'
except Exception:
    ram = 'RAM n/a'
versions = {}
for mod in ['sklearn', 'xgboost', 'lightgbm', 'catboost', 'interpret', 'shap', 'numpy', 'pandas']:
    try: versions[mod] = __import__(mod).__version__
    except Exception: versions[mod] = 'n/a'
results['cost'] = {
    'hardware': f'{platform.processor() or platform.machine()}, {os.cpu_count()} logical cores, {ram}',
    'total_hours': round(total / 3600, 2),
    'tuning_hours': {n: round(v / 3600, 2) for n, v in tune_secs.items()},
    'tuning_share_of_model_time': round(sum(tune_secs.values()) / (sum(tune_secs.values()) + sum(model_secs.values())), 3),
    'minutes_per_PV_fit_and_explain': {n: round(model_secs[n] / M / 60, 2) for n in MODELS},
    'bootstrap_seconds_per_model_PV': round(float(pv_df.t_boot.mean()), 1)}
results['software'] = {'python': platform.python_version(), 'versions': versions,
                       'fold_seed': FOLD_SEED, 'tune_seed': TUNE_SEED}
log('Cost', results['cost']); log('Software', results['software'])

## 16. Save results

In [ ]:
with open(f'{OUT_DIR}/results.json', 'w') as fh:
    json.dump(results, fh, indent=1, default=to_json)
log(f'ALL DONE in {(time.perf_counter() - T0) / 3600:.2f} h this session')